# 😷 Face Mask / Helmet Detection — Complete Pipeline
**Author:** Bilal Ahmed (231980028) — GIFT University
**Model:** MobileNetV2 Transfer Learning
**Task:** Binary Classification (Mask / No Mask)

---
### Steps:
1. Install & Import Libraries
2. Dataset Preparation
3. Data Augmentation & Generators
4. Build MobileNetV2 Model
5. Train (Phase 1 — Head Only)
6. Fine-Tune (Phase 2 — Unfreeze Top Layers)
7. Evaluate — Confusion Matrix, Report
8. Real-time Webcam Demo
9. Streamlit App Launch

In [ ]:
# ─── Cell 1: Install Requirements ────────────────────────────
!pip install tensorflow opencv-python matplotlib seaborn scikit-learn streamlit pillow tqdm -q

In [ ]:
# ─── Cell 2: Imports ─────────────────────────────────────────
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import confusion_matrix, classification_report

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ─── Cell 3: Config ──────────────────────────────────────────
# Change TASK to 'helmet' for helmet detection
TASK        = 'mask'       # 'mask' or 'helmet'
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS_P1   = 20          # Phase 1 (head only)
EPOCHS_P2   = 10          # Phase 2 (fine-tune)
LR_P1       = 1e-4
LR_P2       = 1e-5

TASK_CONFIG = {
    'mask': {
        'train_dir'  : 'dataset/mask/train',
        'val_dir'    : 'dataset/mask/val',
        'class_names': ['with_mask', 'without_mask'],
        'model_path' : 'models/mask_model.h5',
    },
    'helmet': {
        'train_dir'  : 'dataset/helmet/train',
        'val_dir'    : 'dataset/helmet/val',
        'class_names': ['helmet', 'no_helmet'],
        'model_path' : 'models/helmet_model.h5',
    },
}

config      = TASK_CONFIG[TASK]
CLASS_NAMES = config['class_names']
MODEL_PATH  = config['model_path']

os.makedirs('models', exist_ok=True)
print(f'Task: {TASK.upper()}')
print(f'Classes: {CLASS_NAMES}')

In [ ]:
# ─── Cell 4: Data Generators ─────────────────────────────────
# Training: with augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    zoom_range=0.2,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
)

# Validation: only preprocessing
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    config['train_dir'],
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
)

val_gen = val_datagen.flow_from_directory(
    config['val_dir'],
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

print(f'Train samples: {train_gen.samples}')
print(f'Val   samples: {val_gen.samples}')
print(f'Class indices: {train_gen.class_indices}')

In [ ]:
# ─── Cell 5: Visualize Sample Images ─────────────────────────
def show_samples(generator, class_names, n=8):
    images, labels = next(generator)
    generator.reset()
    
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    fig.suptitle('Sample Training Images (After Augmentation)', fontsize=14, fontweight='bold')
    
    for i, ax in enumerate(axes.flatten()):
        if i < n:
            img = (images[i] + 1.0) / 2.0   # de-normalize
            img = np.clip(img, 0, 1)
            ax.imshow(img)
            ax.set_title(class_names[int(labels[i])], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(train_gen, CLASS_NAMES)

In [ ]:
# ─── Cell 6: Build MobileNetV2 Model ─────────────────────────
def build_model(input_shape=(224, 224, 3), lr=1e-4):
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False   # Freeze base
    
    inputs  = tf.keras.Input(shape=input_shape)
    x       = base_model(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(128, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model()
model.summary()

total = sum(tf.size(w).numpy() for w in model.weights)
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f'\nTotal params    : {total:,}')
print(f'Trainable params: {trainable:,}')
print(f'Frozen params   : {total-trainable:,}')

In [ ]:
# ─── Cell 7: Phase 1 — Train Head Only ───────────────────────
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(MODEL_PATH, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-7, verbose=1),
]

print('\nPhase 1: Training classification head (base frozen)...')
history = model.fit(
    train_gen,
    epochs=EPOCHS_P1,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1,
)

print(f'\nBest Val Accuracy: {max(history.history["val_accuracy"])*100:.2f}%')

In [ ]:
# ─── Cell 8: Phase 2 — Fine-Tuning ───────────────────────────
print('Phase 2: Unfreezing top layers of MobileNetV2...')

base_model = model.layers[1]   # MobileNetV2 is layer index 1
base_model.trainable = True

# Freeze first 100 layers, unfreeze rest
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with lower LR
model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_P2),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_ft = model.fit(
    train_gen,
    epochs=EPOCHS_P2,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1,
)

# Merge histories
for key in history.history:
    history.history[key].extend(history_ft.history[key])

model.save(MODEL_PATH)
print(f'\nModel saved → {MODEL_PATH}')

In [ ]:
# ─── Cell 9: Plot Training Curves ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History', fontsize=16, fontweight='bold')

axes[0].plot(history.history['accuracy'],     label='Train', color='#2196F3', lw=2)
axes[0].plot(history.history['val_accuracy'], label='Val',   color='#4CAF50', lw=2)
axes[0].axvline(x=EPOCHS_P1, color='orange', linestyle='--', label='Fine-tune start')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

axes[1].plot(history.history['loss'],     label='Train', color='#F44336', lw=2)
axes[1].plot(history.history['val_loss'], label='Val',   color='#FF9800', lw=2)
axes[1].axvline(x=EPOCHS_P1, color='orange', linestyle='--', label='Fine-tune start')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'models/{TASK}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Cell 10: Evaluate — Confusion Matrix ────────────────────
val_gen.reset()
preds_prob = model.predict(val_gen, verbose=1).flatten()
y_true = val_gen.classes
y_pred = (preds_prob >= 0.5).astype(int)

accuracy = np.mean(y_true == y_pred) * 100
print(f'\n✅ Validation Accuracy: {accuracy:.2f}%')

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, linewidths=0.5)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'models/{TASK}_confusion_matrix.png', dpi=150)
plt.show()

# Classification report
print('\nClassification Report:')
print('─' * 60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# ─── Cell 11: Sample Predictions ─────────────────────────────
val_gen.reset()
images, labels = next(val_gen)
preds = (model.predict(images, verbose=0).flatten() >= 0.5).astype(int)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Sample Predictions  (Green=Correct | Red=Wrong)', fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    if i < len(images):
        img = (images[i] + 1.0) / 2.0
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        pred_label   = CLASS_NAMES[preds[i]]
        actual_label = CLASS_NAMES[int(labels[i])]
        correct = preds[i] == int(labels[i])
        ax.set_title(f'P: {pred_label}\nA: {actual_label}',
                     color='green' if correct else 'red', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'models/{TASK}_sample_predictions.png', dpi=150)
plt.show()

In [ ]:
# ─── Cell 12: Predict on a Single Image ──────────────────────
from PIL import Image

def predict_single(image_path, model, class_names):
    img       = Image.open(image_path).convert('RGB').resize((224, 224))
    arr       = np.array(img, dtype=np.float32)
    arr       = preprocess_input(arr)
    arr       = np.expand_dims(arr, axis=0)
    
    prob      = model.predict(arr, verbose=0)[0][0]
    pred_idx  = int(prob >= 0.5)
    label     = class_names[pred_idx]
    confidence= prob if pred_idx == 1 else 1.0 - prob
    
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis('off')
    is_good = (TASK == 'mask' and label == 'with_mask') or \
              (TASK == 'helmet' and label == 'helmet')
    plt.title(f'Prediction: {label.upper()}\nConfidence: {confidence*100:.1f}%',
              fontsize=13, fontweight='bold',
              color='green' if is_good else 'red')
    plt.show()
    print(f'Label: {label}  |  Confidence: {confidence*100:.1f}%')

# ── Usage: Replace with your image path ──────
# predict_single('path/to/your/image.jpg', model, CLASS_NAMES)

In [ ]:
# ─── Cell 13: Webcam Detection (Run in script, not notebook) ──
# In notebook, webcam won't show GUI window.
# Run this from terminal instead:
#
#   python realtime_detect.py --task mask
#
print('To run real-time detection, use terminal:')
print(f'  python realtime_detect.py --task {TASK}')
print()
print('To launch the web app:')
print('  streamlit run app/streamlit_app.py')

---
## Summary

| Step | Description | Status |
|------|-------------|--------|
| 1 | Dataset Loading + Augmentation | ✅ |
| 2 | MobileNetV2 Model Building | ✅ |
| 3 | Phase 1 Training (frozen base) | ✅ |
| 4 | Phase 2 Fine-tuning | ✅ |
| 5 | Confusion Matrix + Report | ✅ |
| 6 | Sample Predictions Grid | ✅ |
| 7 | Single Image Inference | ✅ |
| 8 | Real-time Detection | ✅ (run in terminal) |
| 9 | Streamlit Web App | ✅ (run in terminal) |

**Bilal Ahmed | 231980028 | GIFT University**